# Thống kê cấu trúc tài liệu

:::{note}
Chương này là **khung thiết kế kiến trúc và lược đồ thực nghiệm dự kiến cho Giai đoạn 2**. Bảng biểu và đồ thị tự động đọc dữ liệu từ `report_book/data/m2_summary.json` và hiển thị ở trạng thái khung mẫu chờ kết quả khi bộ phân tích cú pháp hoàn thành.
:::


## 1. Tổng quan số lượng phần tử

Bảng dưới đây tổng hợp kết quả bóc tách cấu trúc tài liệu LaTeX, xác định đường biên dịch và dựng cây phân cấp trên mọi phiên bản đã thu thập từ Giai đoạn 1.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

with open("../data/m2_summary.json", "r", encoding="utf-8") as f:
    m2_summary = json.load(f)

is_placeholder = m2_summary.get("is_placeholder", False)
p_overview = m2_summary["parsing_overview"]

def fmt_p(val, suffix="", fmt_spec=","):
    if is_placeholder or val is None:
        return "-- (chờ triển khai Giai đoạn 2)"
    if fmt_spec == "percent":
        return f"{val:.2f}%"
    return f"{val:{fmt_spec}}{suffix}"

overview_df = pd.DataFrame([
    {"Chỉ số cấu trúc": "Tổng số bài báo xử lý", "Giá trị thực nghiệm": fmt_p(p_overview['total_publications_processed'])},
    {"Chỉ số cấu trúc": "Tổng số phiên bản xử lý", "Giá trị thực nghiệm": fmt_p(p_overview['total_versions_processed'])},
    {"Chỉ số cấu trúc": "Số phiên bản bóc tách cú pháp thành công", "Giá trị thực nghiệm": fmt_p(p_overview['successful_parsed_versions'])},
    {"Chỉ số cấu trúc": "Số phiên bản dạng bản ghi rỗng", "Giá trị thực nghiệm": fmt_p(p_overview['empty_records_versions'])},
    {"Chỉ số cấu trúc": "Tỉ lệ phân tích cú pháp thành công", "Giá trị thực nghiệm": fmt_p(p_overview['parsing_success_rate_percent'], fmt_spec="percent")}
])
overview_df.style.hide(axis="index").set_caption("Bảng 2.1: Khung tổng hợp kết quả phân tích cú pháp Giai đoạn 2")


### Nhận xét về tỉ lệ bóc tách cú pháp

- Tỉ lệ phân tích cú pháp thành công trên các phiên bản có mã TeX hợp lệ đạt `[...]%`. Thuật toán phát hiện tệp gốc chứa lệnh `\documentclass` và lần theo chuỗi `\input`, `\include` đã xử lý được cấu trúc tệp của các bài báo.
- Toàn bộ `[...]` phiên bản dạng PDF hoặc Word được ghi nhận dưới dạng bản ghi rỗng với 0 phần tử để giữ đúng mẫu số của tập dữ liệu.


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2), dpi=120)
dedup = m2_summary["deduplication_metrics"]

if is_placeholder or dedup["elements_before_dedup"] is None:
    ax1.text(0.5, 0.5, "[Khung đồ thị mẫu: Khử trùng lặp phần tử]\nChờ kết quả xuất bản bảng elements",
             ha="center", va="center", fontsize=9.5, color="#444444",
             bbox=dict(boxstyle="round,pad=0.6", facecolor="#f8f9fa", edgecolor="#cccccc", linestyle="--"))
    ax1.set_xlim(0, 1)
    ax1.set_ylim(0, 1)
    ax1.set_yticks([])
    ax1.set_xticks([0.3, 0.7])
    ax1.set_xticklabels(["Trước khử trùng lặp", "Sau khử trùng lặp"])
    
    ax2.text(0.5, 0.5, "[Khung đồ thị mẫu: Khử trùng lặp BibTeX]\nChờ kết quả hợp nhất bảng bibtex",
             ha="center", va="center", fontsize=9.5, color="#444444",
             bbox=dict(boxstyle="round,pad=0.6", facecolor="#f8f9fa", edgecolor="#cccccc", linestyle="--"))
    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1)
    ax2.set_yticks([])
    ax2.set_xticks([0.3, 0.7])
    ax2.set_xticklabels(["Trước khử trùng lặp", "Sau khử trùng lặp"])
else:
    pass

ax1.set_title("Hình 2.1a: Số lượng phần tử nội dung", fontsize=11)
ax2.set_title("Hình 2.1b: Số lượng mục trích dẫn", fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
dedup = m2_summary["deduplication_metrics"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5), dpi=120)

el_cats = ["Trước khử trùng lặp", "Sau khử trùng lặp\n(Bảng elements)"]
el_vals = [dedup["elements_before_dedup"] / 1e6, dedup["elements_after_dedup"] / 1e6]
ax1.bar(el_cats, el_vals, color=["#ff7f00", "#377eb8"], width=0.45, edgecolor="black", linewidth=0.5)
ax1.set_ylabel("Triệu phần tử", fontsize=10)
ax1.set_title("Hình 2.1a: Số lượng phần tử nội dung", fontsize=11)
for i, v in enumerate(el_vals):
    ax1.text(i, v + 0.1, f"{v:.2f} M", ha="center", va="bottom", fontsize=10)
ax1.text(0.5, max(el_vals)*0.7, f"Giảm: {dedup['element_reduction_percent']:.1f}%", fontsize=11, ha="center",
         bbox=dict(boxstyle="round,pad=0.4", facecolor="#fff2cc", edgecolor="#d6b656"))
ax1.set_ylim(0, max(el_vals) * 1.25)

bib_cats = ["Trước khử trùng lặp", "Sau khử trùng lặp\n(Bảng bibtex)"]
bib_vals = [dedup["bibtex_entries_before_dedup"] / 1e3, dedup["bibtex_entries_after_dedup"] / 1e3]
ax2.bar(bib_cats, bib_vals, color=["#e41a1c", "#4daf4a"], width=0.45, edgecolor="black", linewidth=0.5)
ax2.set_ylabel("Nghìn mục trích dẫn", fontsize=10)
ax2.set_title("Hình 2.1b: Số lượng mục trích dẫn", fontsize=11)
for i, v in enumerate(bib_vals):
    ax2.text(i, v + 15, f"{v:.1f} K", ha="center", va="bottom", fontsize=10)
ax2.text(0.5, max(bib_vals)*0.7, f"Giảm: {dedup['bibtex_reduction_percent']:.1f}%", fontsize=11, ha="center",
         bbox=dict(boxstyle="round,pad=0.4", facecolor="#d5e8d4", edgecolor="#82b366"))
ax2.set_ylim(0, max(bib_vals) * 1.25)

plt.tight_layout()
plt.show()


kinds = m2_summary["hierarchy_distribution"]["element_kinds"]
fig, ax = plt.subplots(figsize=(7, 4.2), dpi=120)

if is_placeholder or kinds["sentence"] is None:
    ax.text(0.5, 0.5, "[Khung đồ thị mẫu: Phân bố các dạng nút lá trên cây phân cấp]\nChờ kết quả dựng cây tài liệu Giai đoạn 2",
            ha="center", va="center", fontsize=10, color="#444444",
            bbox=dict(boxstyle="round,pad=0.8", facecolor="#f8f9fa", edgecolor="#cccccc", linestyle="--"))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_xticks([])
else:
    pass

ax.set_title("Hình 2.2: Tỉ lệ phân bố các dạng nút lá trên cây phân cấp", fontsize=12, pad=12)
plt.tight_layout()
plt.show()


## 3. Phân bố cấp bậc và dạng nút lá

Cây phân cấp của các bài báo có độ sâu trung bình đạt `[...]` tầng từ gốc tài liệu đến các nút lá. Đồ thị dưới đây thể hiện tỉ lệ của các dạng nút lá trên cây phân cấp.


In [ ]:
errs = m2_summary["parse_errors_analysis"]
fig, ax = plt.subplots(figsize=(8, 3.8), dpi=120)

if is_placeholder or errs["unconventional_macros"] is None:
    ax.text(0.5, 0.5, "[Khung đồ thị mẫu: Phân loại các nguyên nhân lỗi cú pháp LaTeX]\nChờ kết quả phân tích ngoại lệ cú pháp Giai đoạn 2",
            ha="center", va="center", fontsize=10, color="#444444",
            bbox=dict(boxstyle="round,pad=0.8", facecolor="#f8f9fa", edgecolor="#cccccc", linestyle="--"))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_xticks([])
else:
    pass

ax.set_title("Hình 2.3: Phân loại các nguyên nhân lỗi cú pháp LaTeX", fontsize=12, pad=12)
plt.tight_layout()
plt.show()


### Nhận xét về cơ cấu nút lá

- Các câu văn xuôi chiếm `[...]%`, là phần nội dung chính mang ngữ cảnh và chứa các câu lệnh `\cite`.
- Các khối công thức toán chiếm `[...]%`, thể hiện đặc thù của các bài báo thuộc ngành máy tính và toán học trên arXiv.
- Các nút dạng hình ảnh và bảng biểu chiếm `[...]%`. Do giữ nguyên các lệnh `figure` và `\includegraphics`, cấu trúc chú thích và vị trí của hình ảnh được bảo tồn trên cây.


## 4. Các trường hợp ngoại lệ

Trong quá trình xử lý mã TeX từ nhiều tác giả, bộ phân tích cú pháp đã gặp một số dạng bất thường về cú pháp.

In [ ]:
errs = m2_summary["parse_errors_analysis"]

fig, ax = plt.subplots(figsize=(8, 4), dpi=120)
err_labels = [
    "Lệnh tự định nghĩa lồng nhau",
    "Lệch dấu ngoặc hoặc dấu toán",
    "Định dạng bibitem không chuẩn",
    "Ký tự bảng mã đặc thù"
]
err_vals = [errs["unconventional_macros"], errs["unbalanced_delimiters"], errs["non_standard_bib_style"], errs["encoding_and_foreign_chars"]]

bars = ax.barh(err_labels, err_vals, color="#bc80bd", edgecolor="black", linewidth=0.5)
ax.set_xlabel("Tỉ lệ trong tổng số trường hợp cảnh báo (%)", fontsize=10)
ax.set_title("Hình 2.3: Phân loại các nguyên nhân lỗi cú pháp LaTeX", fontsize=12, pad=12)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.8, bar.get_y() + bar.get_height()/2.0, f"{w:.1f}%", ha="left", va="center", fontsize=9)
ax.set_xlim(0, max(err_vals) * 1.2)
ax.invert_yaxis()
plt.tight_layout()
plt.show()


Phân tích sơ bộ từ mẫu thử nghiệm cho thấy:

- Lệnh tự định nghĩa chiếm `[...]%`: Một số tác giả dùng lệnh `\def` để viết tắt công thức. Bộ phân tích cú pháp thay thế các macro cơ bản trước khi phân tích để đưa về dạng chuẩn.
- Lệch dấu ngoặc chiếm `[...]%`: Một số bài báo thiếu dấu ngoặc nhọn đóng hoặc dùng lẫn giữa `$ ... $` và `\begin{math}`. Bộ phân tích cú pháp dùng cơ chế kiểm tra dấu theo dòng để tránh lỗi dừng chương trình.